<a href="https://colab.research.google.com/github/Eddiegah/SymFM/blob/master/SymFM_Sensitivity_Colab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SymFM: Hyperparameter Sensitivity Analysis
**Author:** Edmund Eric Gah

Tests lambda2 (physics weight) and lambda3 (sparsity weight)
on Lorenz-96 N=20. Produces the 3x3 sensitivity table for the paper.

**Before running:** Runtime > Change runtime type > T4 GPU

In [ ]:
!pip install torch numpy scipy -q
import torch
import numpy as np
import json
import os
import time
import torch.nn as nn
from scipy.integrate import solve_ivp
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)
print('Setup complete.')

Device: cuda
GPU: Tesla T4
Setup complete.


In [ ]:
# Generate Lorenz-96 data at N=20
def lorenz96(t, x, F=8.0):
    N = len(x)
    dxdt = np.zeros(N)
    for i in range(N):
        dxdt[i] = (x[(i+1)%N] - x[(i-2)%N]) * x[(i-1)%N] - x[i] + F
    return dxdt

def generate_data(N, seed=42):
    np.random.seed(seed)
    t_eval = np.arange(0.0, 20.0, 0.01)
    T = len(t_eval)
    X_all  = np.zeros((20, T, N))
    dX_all = np.zeros((20, T, N))
    for i in range(20):
        x0 = 8.0 * np.ones(N)
        x0[0] += 0.01 * np.random.randn()
        x0 += 0.5 * np.random.randn(N)
        sol = solve_ivp(
            lambda t, x: lorenz96(t, x),
            (0, 20), x0, method='RK45',
            t_eval=t_eval, rtol=1e-8, atol=1e-8
        )
        if sol.success:
            X_all[i] = sol.y.T
            dX_all[i] = np.array([
                lorenz96(t_eval[k], X_all[i, k]) for k in range(T)
            ])
    return X_all, dX_all

print('Generating Lorenz-96 N=20...')
X_all, dX_all = generate_data(20)
print(f'Done. Shape: {X_all.shape}')

Generating Lorenz-96 N=20...
Done. Shape: (20, 2000, 20)


In [ ]:
# Define SymFM model components

def huber_loss(pred, target, delta=0.5):
    err = torch.abs(pred - target)
    loss = torch.where(err <= delta, 0.5 * err**2, delta * (err - 0.5 * delta))
    return loss.mean()

def mse_loss(pred, target):
    return ((pred - target)**2).mean()

def lorenz_residual(x, pred, forcing=8.0):
    N = x.shape[1]
    true_deriv = torch.zeros_like(x)
    for i in range(N):
        true_deriv[:, i] = (
            (x[:, (i+1)%N] - x[:, (i-2)%N]) * x[:, (i-1)%N]
            - x[:, i] + forcing
        )
    return mse_loss(pred, true_deriv)

class ActiveSubspace(nn.Module):
    def __init__(self, N, d, eta=0.01):
        super().__init__()
        self.N = N
        self.d = d
        self.eta = eta
        W_init = torch.randn(d, N)
        Q, _ = torch.linalg.qr(W_init.T)
        self.W = nn.Parameter(Q[:, :d].T)

    def forward(self, x):
        return x @ self.W.T

    def ortho_loss(self):
        WWT = self.W @ self.W.T
        eye = torch.eye(self.d, device=self.W.device)
        return self.eta * torch.norm(WWT - eye, p='fro') ** 2

class SymbolicHead(nn.Module):
    def __init__(self, d, N, hidden=128):
        super().__init__()
        self.d = d
        self.N = N
        self.uni = nn.Sequential(
            nn.Linear(d, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, N)
        )
        self.pair = nn.Sequential(
            nn.Linear(d * d, hidden), nn.SiLU(),
            nn.Linear(hidden, N)
        )
        self.comb = nn.Linear(2 * N, N)
        nn.init.xavier_uniform_(self.comb.weight, gain=0.1)
        nn.init.zeros_(self.comb.bias)

    def forward(self, xp):
        u = self.uni(xp)
        b = xp.shape[0]
        outer = (xp.unsqueeze(2) * xp.unsqueeze(1)).view(b, -1)
        p = self.pair(outer)
        return self.comb(torch.cat([u, p], dim=-1))

    def sparsity_loss(self):
        return 0.001 * torch.norm(self.comb.weight, p=1)

print('Model components defined.')

Model components defined.


In [ ]:
# Training function
def run_single_trial(X_data, dX_data, state_dim, subspace_dim,
                     physics_weight, sparsity_weight,
                     num_epochs=1000, learn_rate=3e-4, run_device='cuda'):

    T = X_data.shape[1]
    T_train = int(0.7 * T)
    T_valid  = int(0.15 * T)

    X_train = torch.tensor(X_data[0, :T_train],             dtype=torch.float32).to(run_device)
    y_train = torch.tensor(dX_data[0, :T_train],            dtype=torch.float32).to(run_device)
    X_valid = torch.tensor(X_data[0, T_train:T_train+T_valid], dtype=torch.float32).to(run_device)
    y_valid = torch.tensor(dX_data[0, T_train:T_train+T_valid], dtype=torch.float32).to(run_device)
    X_test  = torch.tensor(X_data[0, T_train+T_valid:],     dtype=torch.float32).to(run_device)
    y_test  = torch.tensor(dX_data[0, T_train+T_valid:],    dtype=torch.float32).to(run_device)

    proj = ActiveSubspace(state_dim, subspace_dim).to(run_device)
    head = SymbolicHead(subspace_dim, state_dim).to(run_device)
    all_params = list(proj.parameters()) + list(head.parameters())

    optimizer = torch.optim.AdamW(all_params, lr=learn_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-6
    )

    best_val_loss = float('inf')
    best_proj_state = None
    best_head_state = None

    for epoch in range(num_epochs):
        proj.train()
        head.train()
        optimizer.zero_grad()

        x_proj = proj(X_train)
        pred   = head(x_proj)

        rec_loss     = huber_loss(pred, y_train)
        phys_loss    = lorenz_residual(X_train, pred)
        sparse_loss  = head.sparsity_loss()
        ortho_loss   = proj.ortho_loss()

        total_loss = (rec_loss
                      + physics_weight * phys_loss
                      + sparsity_weight * sparse_loss
                      + 0.01 * ortho_loss)

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
        optimizer.step()
        scheduler.step()

        proj.eval()
        head.eval()
        with torch.no_grad():
            val_pred = head(proj(X_valid))
            val_loss = huber_loss(val_pred, y_valid)

        if float(val_loss) < best_val_loss:
            best_val_loss = float(val_loss)
            best_proj_state = {k: v.cpu().clone() for k, v in proj.state_dict().items()}
            best_head_state = {k: v.cpu().clone() for k, v in head.state_dict().items()}

    if best_proj_state is not None:
        proj.load_state_dict(best_proj_state)
        head.load_state_dict(best_head_state)

    proj.eval()
    head.eval()
    with torch.no_grad():
        test_pred = head(proj(X_test)).cpu().numpy()

    y_test_np = y_test.cpu().numpy()
    error = float(
        np.linalg.norm(y_test_np - test_pred) /
        (np.linalg.norm(y_test_np) + 1e-10)
    )
    return min(error, 10.0)

print('Training function defined.')

Training function defined.


In [ ]:
# Run 3x3 hyperparameter sensitivity grid
STATE_DIM    = 20
SUBSPACE_DIM = 8

physics_weights  = [1.0, 2.0, 4.0]
sparsity_weights = [0.1, 0.5, 1.0]

result_grid = np.zeros((3, 3))

print('Running 3x3 sensitivity grid on Lorenz-96 N=20')
print('9 experiments x 1000 epochs each')
print()

for row_idx, pw in enumerate(physics_weights):
    for col_idx, sw in enumerate(sparsity_weights):
        label = '  <-- DEFAULT' if (pw == 2.0 and sw == 0.5) else ''
        print(f'Experiment ({row_idx+1},{col_idx+1}): '
              f'physics_weight={pw}, sparsity_weight={sw}{label}')

        err = run_single_trial(
            X_data         = X_all,
            dX_data        = dX_all,
            state_dim      = STATE_DIM,
            subspace_dim   = SUBSPACE_DIM,
            physics_weight = pw,
            sparsity_weight= sw,
            num_epochs     = 1000,
            learn_rate     = 3e-4,
            run_device     = device
        )
        result_grid[row_idx, col_idx] = err
        print(f'  -> L2 Error = {err:.4f}')
        print()

print('All 9 experiments complete.')

Running 3x3 sensitivity grid on Lorenz-96 N=20
9 experiments x 1000 epochs each

Experiment (1,1): physics_weight=1.0, sparsity_weight=0.1
  -> L2 Error = 0.9978

Experiment (1,2): physics_weight=1.0, sparsity_weight=0.5
  -> L2 Error = 1.0038

Experiment (1,3): physics_weight=1.0, sparsity_weight=1.0
  -> L2 Error = 0.9929

Experiment (2,1): physics_weight=2.0, sparsity_weight=0.1
  -> L2 Error = 0.9766

Experiment (2,2): physics_weight=2.0, sparsity_weight=0.5  <-- DEFAULT
  -> L2 Error = 0.9934

Experiment (2,3): physics_weight=2.0, sparsity_weight=1.0
  -> L2 Error = 0.9685

Experiment (3,1): physics_weight=4.0, sparsity_weight=0.1
  -> L2 Error = 0.9710

Experiment (3,2): physics_weight=4.0, sparsity_weight=0.5
  -> L2 Error = 0.9562

Experiment (3,3): physics_weight=4.0, sparsity_weight=1.0
  -> L2 Error = 0.9791

All 9 experiments complete.


In [ ]:
# Print final table
print()
print('=' * 65)
print('HYPERPARAMETER SENSITIVITY: Lorenz-96 N=20, Relative L2 Error')
print('(Lower is better. Default setting marked with **)')
print('=' * 65)
print(f'{"":<20} {"sparsity=0.1":<16} {"sparsity=0.5":<16} {"sparsity=1.0"}')
print('-' * 65)
for row_idx, pw in enumerate(physics_weights):
    row = f'physics={pw:<12}  '
    for col_idx, sw in enumerate(sparsity_weights):
        val = result_grid[row_idx, col_idx]
        marker = '**' if (pw == 2.0 and sw == 0.5) else '  '
        row += f'{val:.4f}{marker}        '
    print(row)
print('=' * 65)
print()

# Find best setting
best_idx = np.unravel_index(np.argmin(result_grid), result_grid.shape)
best_pw  = physics_weights[best_idx[0]]
best_sw  = sparsity_weights[best_idx[1]]
best_val = result_grid[best_idx]
print(f'Best setting: physics_weight={best_pw}, sparsity_weight={best_sw}')
print(f'Best L2 Error: {best_val:.4f}')
print(f'Range: {result_grid.min():.4f} to {result_grid.max():.4f}')

# Save
sensitivity_data = {
    'physics_weights':  physics_weights,
    'sparsity_weights': sparsity_weights,
    'grid':             result_grid.tolist(),
    'best_physics':     float(best_pw),
    'best_sparsity':    float(best_sw),
    'best_l2':          float(best_val),
    'N':                STATE_DIM,
    'metric':           'Relative L2 Error'
}
with open('results/sensitivity_results.json', 'w') as f:
    json.dump(sensitivity_data, f, indent=2)
print('Saved: results/sensitivity_results.json')


HYPERPARAMETER SENSITIVITY: Lorenz-96 N=20, Relative L2 Error
(Lower is better. Default setting marked with **)
                     sparsity=0.1     sparsity=0.5     sparsity=1.0
-----------------------------------------------------------------
physics=1.0           0.9978          1.0038          0.9929          
physics=2.0           0.9766          0.9934**        0.9685          
physics=4.0           0.9710          0.9562          0.9791          

Best setting: physics_weight=4.0, sparsity_weight=0.5
Best L2 Error: 0.9562
Range: 0.9562 to 1.0038
Saved: results/sensitivity_results.json


In [ ]:
import shutil
shutil.make_archive('SymFM_sensitivity_results', 'zip', 'results')
print('Download: SymFM_sensitivity_results.zip from Files panel on the left.')

Download: SymFM_sensitivity_results.zip from Files panel on the left.
